In [ ]:
!pip install transformers sentencepiece torch tqdm ipywidgets sacremoses
!pip install transliterate

  Using cached attrs-25.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached isodate-0.7.2-py3-none-any.whl.metadata (11 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
Using cached attrs-25.4.0-py3-none-any.whl (67 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 2.8 MB/s  0:00:04eta 0:00:010m
Using cached isodate-0.7.2-py3-none-any.whl (22 kB)
Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.2/587.2 kB 4.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17/17 [phonemizer]7 [jsonschema]


In [1]:
import pandas as pd
import numpy as np

def clean_from_unknown(df: pd.DataFrame, column_name: str) -> pd.DataFrame:
    df = df.copy()
    """Removes rows with 'unknown' values in any column."""
    df[column_name] = df[column_name].apply(
        lambda x: np.nan
        if pd.isna(x) or str(x).strip().lower() == "unknown"
        else x
    )
    return df


In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import MarianMTModel, MarianTokenizer
from tqdm.auto import tqdm

# модель перевода
_MODEL_NAME = "Helsinki-NLP/opus-mt-en-ru"
_tokenizer = MarianTokenizer.from_pretrained(_MODEL_NAME)
_model = MarianMTModel.from_pretrained(_MODEL_NAME)

def translate_dataset_to_ru(df: pd.DataFrame) -> pd.DataFrame:
    """
    Translates selected perfume columns from EN to RU.
    Gender is ignored completely.
    NaN stays NaN.
    Returns new DataFrame.
    """

    cols_to_translate = [
        "Perfume", "Brand", "Country",
        "Top", "Middle", "Base",
        "mainaccord1", "mainaccord2", "mainaccord3",
        "mainaccord4", "mainaccord5",
    ]

    df = df.copy()

    for col in cols_to_translate:
        if col not in df.columns:
            continue

        series = df[col]
        mask = series.notna()
        unique_texts = series[mask].unique()

        translations = {}
        
        batch_size = 16
        total_batches = (len(unique_texts) + batch_size - 1) // batch_size

        for i in tqdm(range(0, len(unique_texts), batch_size), 
                      desc=f"Translating {col}", 
                      total=total_batches):
            batch = unique_texts[i:i+batch_size].tolist()
            encoded = _tokenizer(batch, return_tensors="pt", padding=True, truncation=True)

            with torch.no_grad():
                generated = _model.generate(**encoded)

            decoded = _tokenizer.batch_decode(generated, skip_special_tokens=True)
            translations.update(dict(zip(batch, decoded)))

        df[col] = series.map(translations)

    return df


In [2]:
def translate_gender_to_ru(df: pd.DataFrame) -> pd.DataFrame:
    """
    Translates gender column from EN to RU. 
    'unisex' -> 'унисекс'
    NaN stays NaN.
    'men' -> 'мужской'
    'women' -> 'женский'
    Returns new DataFrame.
    """
    
    df = df.copy()
    gender_map = {
        "men" : "мужской",
        "women" : "женский",
        "unisex" : "унисекс",
    }
    df["Gender_ru"] = df["Gender"].map(gender_map)
    return df

In [3]:
df = pd.read_csv(
    "../data/raw/fra_cleaned.csv",
    encoding="latin-1",
    sep=";"
)

df = clean_from_unknown(df, "Perfumer1")
df = translate_dataset_to_ru(df)
df = translate_gender_to_ru(df)
df.to_csv("../data/processed/dataset_translated.csv", index=False, encoding="utf-8")

NameError: name 'translate_dataset_to_ru' is not defined

Translate to Russian

In [4]:
df = pd.read_csv(
    "../data/processed/dataset_translated.csv",
)



In [5]:
from transliterate import translit
import pandas as pd

def perfume_to_ru(x: str) -> str:
    if pd.isna(x):
        return x

    x = str(x).strip()
    if not x:
        return x

    try:
        return translit(x, 'ru')
    except Exception:
        return x



In [6]:
df_en = pd.read_csv(
    "../data/raw/fra_cleaned.csv",
    encoding="latin-1",
    sep=";"
)

# Drop columns that will be renamed or are duplicates
df = df.drop(columns=["Perfume", "Gender_ru"], errors="ignore")

df_final = df_en.merge(
    df,
    on="url",
    how="inner",
    validate="one_to_one"
)

# Drop _y columns that are identical to _x (Year, Rating Value, Rating Count, Perfumer1, Perfumer2)
df_final = df_final.drop(columns=["Year_y", "Rating Value_y", "Rating Count_y", "Perfumer1_y", "Perfumer2_y"], errors="ignore")

# Rename _x columns to _en (English)
rename_x_to_en = {
    "Brand_x": "Brand_en",
    "Country_x": "Country_en",
    "Gender_x": "Gender_en",
    "Year_x": "Year",
    "Rating Value_x": "Rating Value",
    "Rating Count_x": "Rating Count",
    "Perfumer1_x": "Perfumer1",
    "Perfumer2_x": "Perfumer2",
    "Top_x": "Top_en",
    "Middle_x": "Middle_en",
    "Base_x": "Base_en",
    "mainaccord1_x": "mainaccord1_en",
    "mainaccord2_x": "mainaccord2_en",
    "mainaccord3_x": "mainaccord3_en",
    "mainaccord4_x": "mainaccord4_en",
    "mainaccord5_x": "mainaccord5_en"
}
df_final = df_final.rename(columns=rename_x_to_en)

# Rename _y columns to _ru (Russian translations)
rename_y_to_ru = {
    "Brand_y": "Brand_ru",
    "Country_y": "Country_ru",
    "Gender_y": "Gender_ru",
    "Top_y": "Top_ru",
    "Middle_y": "Middle_ru",
    "Base_y": "Base_ru",
    "mainaccord1_y": "mainaccord1_ru",
    "mainaccord2_y": "mainaccord2_ru",
    "mainaccord3_y": "mainaccord3_ru",
    "mainaccord4_y": "mainaccord4_ru",
    "mainaccord5_y": "mainaccord5_ru"
}
df_final = df_final.rename(columns=rename_y_to_ru)

df_final.to_csv("../data/processed/dataset_final.csv", index=False, encoding="utf-8")

df = pd.read_csv(
    "../data/processed/dataset_final.csv",
)  
df.head()

,url,Perfume,Brand_en,Country_en,Gender_en,Rating Value,Rating Count,Year,Top_en,Middle_en,...,Country_ru,Gender_ru,Top_ru,Middle_ru,Base_ru,mainaccord1_ru,mainaccord2_ru,mainaccord3_ru,mainaccord4_ru,mainaccord5_ru
0,https://www.fragrantica.com/perfume/xerjoff/ac...,accento-overdose-pride-edition,xerjoff,Italy,unisex,"1,42",201,2022.0,"fruity notes, aldehydes, green notes","bulgarian rose, egyptian jasmine, lily-of-the-...",...,Италия,unisex,"Фруктовые ноты, альдегиды, зелёные ноты","Булгарская роза, египтианская жасмина, лилия-в...","Эвкалипт, сосна",роза,вуаля,Фрукты,ароматические,Цветок
1,https://www.fragrantica.com/perfume/jean-paul-...,classique-pride-2024,jean-paul-gaultier,France,women,"1,86",70,2024.0,"yuzu, citruses","orange blossom, neroli",...,Франция,women,"Юзу, цитрусовые","апельсиновый цветок, нероли","мускус, блондинка",цитрусовые,белый цветок,сладкий,свежие,Муски
2,https://www.fragrantica.com/perfume/jean-paul-...,classique-pride-2023,jean-paul-gaultier,France,unisex,"1,91",285,2023.0,"blood orange, yuzu","neroli, orange blossom",...,Франция,unisex,"Кровь оранжевая, юдзу","Нероли, оранжевый цветок","мускус, белый лес",цитрусовые,белый цветок,сладкий,свежее острое,Муски
3,https://www.fragrantica.com/perfume/bruno-bana...,pride-edition-man,bruno-banani,Germany,men,"1,92",59,2019.0,"guarana, grapefruit, red apple","walnut, lavender, guava",...,Германия,men,"Гуарана, грейпфруты, красное яблоко","Орех, лаванда, гуава","Ветер, бензуин, автожелтый",Фрукты,Сумасшедший,вуаля,тропические,NaN
4,https://www.fragrantica.com/perfume/jean-paul-...,le-male-pride-collector,jean-paul-gaultier,France,men,"1,93",632,2020.0,"mint, lavender, cardamom, artemisia, bergamot","caraway, cinnamon, orange blossom",...,Франция,men,"Мята, лаванда, кардамом, артемизия, бергамот",":: автомобиль, корица, цветок оранжевого цвета","ваниль, сандальная древесина, автожелтый, кедр...",ароматические,тёплый,свежее острое,Корица,ванильный


In [7]:
df = df.drop(columns=["Perfume_ru"], errors="ignore")
df.to_csv("../data/processed/dataset_final.csv", index=False, encoding="utf-8")  
